In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import os
from chromadb.utils import embedding_functions
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
    
    def embed_texts(self, texts: List[str]) -> np.ndarray:

        if not self.model:
            raise ValueError("Model not loaded. Cannot compute embeddings.")
        print(f"Computing embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded. Cannot get embedding dimension.")
        return self.model.get_embedding_dimension()
    
    embedding_manager = EmbeddingManager()
    embedding_manager
    

c:\Users\maddi\OneDrive\Documents\VisualStudio-Git\AgenticAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maddi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4283.38

Model 'all-MiniLM-L6-v2' loaded successfully. Embedding Dimension: 384


In [ ]:
class vector_store:
    def __init__(self, collection_name: str = "pdf documents", persist_directory: str = "../../data/vectorstore-chromadb"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.get_or_create_collection(
                name = self.collection_name,
                metadata = {'description': 'Collection for PDF document embeddings'}
            )
            print(f"Vector Storage initated Collection '{self.collection_name}', loaded successfully.")
            print(f"Exisiting Documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error Initializing ChromaDB: {e}")
            raise
    
